In [4]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
df = pd.read_excel(r"C:\Users\ODAMA\Downloads\02_GTBank_EDA.xlsx",
                   sheet_name='Customer Data',
                   header=2)

# ── BASIC EXPLORATION (print these to understand your data) ───────────────────
print("Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
print("\nBasic Statistics:\n", df.describe())

# ── KPIs ──────────────────────────────────────────────────────────────────────
total_customers     = len(df)
avg_income          = df['Monthly Income (₦)'].mean()
avg_balance         = df['Account Balance (₦)'].mean()
avg_satisfaction    = df['Satisfaction Score'].mean()

# ── CHART DATA ────────────────────────────────────────────────────────────────

# 1. Customer count by Age Group
age_dist = df.groupby('Age Group')['Customer ID'].count().reset_index()
age_dist.columns = ['Age Group', 'Count']
age_order = ['18-25','26-35','36-45','46-55','56+']
age_dist['Age Group'] = pd.Categorical(age_dist['Age Group'], categories=age_order, ordered=True)
age_dist = age_dist.sort_values('Age Group')

# 2. Average Income by Occupation
income_by_job = df.groupby('Occupation')['Monthly Income (₦)'].mean().reset_index().sort_values('Monthly Income (₦)')

# 3. Account Type distribution
account_dist = df.groupby('Account Type')['Customer ID'].count().reset_index()
account_dist.columns = ['Account Type', 'Count']

# 4. Satisfaction Score by State
sat_by_state = df.groupby('State')['Satisfaction Score'].mean().reset_index().sort_values('Satisfaction Score')

# 5. Income vs Account Balance (Scatter)
# used directly from df

# 6. Gender distribution
gender_dist = df.groupby('Gender')['Customer ID'].count().reset_index()
gender_dist.columns = ['Gender', 'Count']

# ── CHARTS ────────────────────────────────────────────────────────────────────

fig_age = px.bar(age_dist, x='Age Group', y='Count',
                 color='Count', color_continuous_scale='Blues',
                 title='CUSTOMERS BY AGE GROUP')

fig_income = px.bar(income_by_job, x='Monthly Income (₦)', y='Occupation',
                    orientation='h', color='Monthly Income (₦)',
                    color_continuous_scale='Blues',
                    title='AVG INCOME BY OCCUPATION')

fig_account = px.pie(account_dist, names='Account Type', values='Count',
                     hole=0.5,
                     color_discrete_sequence=['#08306b','#2171b5','#6baed6','#c6dbef'],
                     title='ACCOUNT TYPE DISTRIBUTION')

fig_sat = px.bar(sat_by_state, x='Satisfaction Score', y='State',
                 orientation='h', color='Satisfaction Score',
                 color_continuous_scale='Blues',
                 title='AVG SATISFACTION SCORE BY STATE')

fig_scatter = px.scatter(df, x='Monthly Income (₦)', y='Account Balance (₦)',
                         color='Account Type',
                         color_discrete_sequence=['#08306b','#2171b5','#6baed6','#c6dbef'],
                         title='INCOME vs ACCOUNT BALANCE')

fig_gender = px.pie(gender_dist, names='Gender', values='Count',
                    hole=0.5,
                    color_discrete_sequence=['#1f3b5c','#6baed6'],
                    title='GENDER DISTRIBUTION')

# ── STYLE ALL CHARTS ──────────────────────────────────────────────────────────
for fig in [fig_age, fig_income, fig_account, fig_sat, fig_scatter, fig_gender]:
    fig.update_layout(
        plot_bgcolor='#eaf3fb',
        paper_bgcolor='#ffffff',
        font=dict(color='#1f3b5c', family='Arial'),
        title_font=dict(size=12, color='#1f3b5c'),
        margin=dict(l=10, r=10, t=35, b=10),
        height=250,
        showlegend=True
    )

# ── STYLES ────────────────────────────────────────────────────────────────────
BLUE_DARK  = '#1f3b5c'
BLUE_MID   = '#2171b5'
BLUE_LIGHT = '#d6e6f2'
WHITE      = '#ffffff'

kpi_card = {
    'backgroundColor': WHITE, 'padding': '8px 14px', 'borderRadius': '8px',
    'textAlign': 'center', 'flex': '1', 'margin': '4px',
    'border': f'1px solid {BLUE_LIGHT}', 'boxShadow': '2px 2px 5px rgba(0,0,0,0.08)'
}
chart_box = {
    'backgroundColor': WHITE, 'padding': '6px', 'borderRadius': '8px',
    'boxShadow': '2px 2px 5px rgba(0,0,0,0.08)', 'flex': '1'
}
sidebar_label = {
    'backgroundColor': BLUE_MID, 'color': WHITE, 'padding': '4px 8px',
    'borderRadius': '4px', 'marginBottom': '5px', 'fontSize': '12px'
}

# ── APP LAYOUT ────────────────────────────────────────────────────────────────
app = Dash(__name__)

app.layout = html.Div(
    style={'backgroundColor': '#eaf3f9', 'padding': '10px', 'fontFamily': 'Arial'},
    children=[

        # Title + KPI Row
        html.Div([
            html.Div(
                html.H3('GTBANK NIGERIA — CUSTOMER EDA DASHBOARD',
                        style={'color': WHITE, 'margin': '0', 'fontSize': '15px'}),
                style={'backgroundColor': BLUE_DARK, 'padding': '10px 16px',
                       'borderRadius': '8px', 'flex': '2', 'marginRight': '8px'}
            ),
            html.Div([
                html.Div([html.P('TOTAL CUSTOMERS', style={'margin':'0','fontSize':'10px','color':BLUE_MID}),
                          html.H5(f'{total_customers:,}',  style={'margin':'0','color':BLUE_DARK,'fontSize':'12px'})], style=kpi_card),
                html.Div([html.P('AVG INCOME',      style={'margin':'0','fontSize':'10px','color':BLUE_MID}),
                          html.H5(f'₦{avg_income:,.0f}',   style={'margin':'0','color':BLUE_DARK,'fontSize':'12px'})], style=kpi_card),
                html.Div([html.P('AVG BALANCE',     style={'margin':'0','fontSize':'10px','color':BLUE_MID}),
                          html.H5(f'₦{avg_balance:,.0f}',  style={'margin':'0','color':BLUE_DARK,'fontSize':'12px'})], style=kpi_card),
                html.Div([html.P('AVG SATISFACTION',style={'margin':'0','fontSize':'10px','color':BLUE_MID}),
                          html.H5(f'{avg_satisfaction:.1f} / 10', style={'margin':'0','color':BLUE_DARK,'fontSize':'12px'})], style=kpi_card),
            ], style={'display': 'flex', 'flex': '4'})
        ], style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '8px'}),

        # Body: Sidebar + Charts
        html.Div([

            # Sidebar
            html.Div([
                html.P('Age Groups', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'4px','fontSize':'11px'}),
                *[html.Div(a, style=sidebar_label) for a in ['18-25','26-35','36-45','46-55','56+']],
                html.Br(),
                html.P('Gender', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'4px','fontSize':'11px'}),
                *[html.Div(g, style=sidebar_label) for g in ['Male','Female']],
                html.Br(),
                html.P('Account Types', style={'color':BLUE_MID,'fontWeight':'bold','marginBottom':'4px','fontSize':'11px'}),
                *[html.Div(a, style=sidebar_label) for a in ['Savings','Current','Salary','Fixed Deposit']],
                html.Br(),
                html.P('Prepared by', style={'fontSize':'10px','color':BLUE_DARK,'margin':'0'}),
                html.P('Odama Joseph', style={'fontSize':'10px','color':BLUE_DARK,'fontWeight':'bold','margin':'0'}),
            ], style={
                'width': '140px', 'backgroundColor': WHITE, 'padding': '10px',
                'borderRadius': '8px', 'boxShadow': '2px 2px 5px rgba(0,0,0,0.08)',
                'marginRight': '8px'
            }),

            # Charts Grid
            html.Div([
                # Row 1
                html.Div([
                    html.Div(dcc.Graph(figure=fig_age,     config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_income,  config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_account, config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '8px'}),

                # Row 2
                html.Div([
                    html.Div(dcc.Graph(figure=fig_sat,     config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_scatter, config={'displayModeBar': False}), style=chart_box),
                    html.Div(dcc.Graph(figure=fig_gender,  config={'displayModeBar': False}), style=chart_box),
                ], style={'display': 'flex', 'gap': '8px', 'marginTop': '8px'}),

            ], style={'flex': '1'})

        ], style={'display': 'flex', 'alignItems': 'flex-start'})
    ]
)

# ── RUN ───────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    app.run(debug=True,port=8051)

Shape: (300, 13)

Missing Values:
 Customer ID                 0
Age Group                   0
Gender                      0
State                       0
Education                   0
Occupation                  0
Account Type                0
Monthly Income (₦)          0
Monthly Transactions (₦)    0
Account Balance (₦)         0
Loan Amount (₦)             0
Satisfaction Score          0
Years with Bank             0
dtype: int64

Basic Statistics:
        Monthly Income (₦)  Monthly Transactions (₦)  Account Balance (₦)  \
count        3.000000e+02              3.000000e+02         3.000000e+02   
mean         5.977621e+05              3.265890e+05         2.364641e+06   
std          3.449834e+05              2.421894e+05         1.472140e+06   
min          5.316075e+04              1.295181e+04         1.741906e+04   
25%          2.988559e+05              1.288977e+05         1.029793e+06   
50%          5.822313e+05              2.690363e+05         2.300287e+06   
75%       